# MPA Temperature & Bleaching Time Series

Generates one figure per Belizean MPA showing:
- Daily CRW SST with DHW-based heat-stress shading
- Monthly climatological mean SST (CRW)
- Monthly ECOSTRESS LST composites
- MMM bleaching threshold
- Bleaching survey distributions (boxplots)

## Data requirements
Place the following in a `data/` directory:
- `dhw_5km.nc` — NOAA CRW 5 km SST/DHW product
- `ct5km_climatology_v3.1.nc` — CRW climatology
- `bleaching_confusion_labeled.csv` — bleaching survey data
- `ecostress_monthly/` — folder of `masked_ECOSTRESS_LST_Median_*.tif` monthly composites
- `Belize_MPAs_Reprojected.shp` (+ sidecar files) — Belize MPA shapefile


In [ ]:
# Install additional dependencies (only needed once)
# !pip install xarray netCDF4 scipy pandas matplotlib rioxarray


In [ ]:
# ================================================================
#  MPA TEMPERATURE & BLEACHING TIME SERIES
#  Per MPA: CRW SST, ECOSTRESS LST, MMM climatology,
#           DHW heat-stress shading, bleaching survey distributions
# ================================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from shapely.geometry import mapping
from shapely.affinity import translate

# ── PATHS ─────────────────────────────────────────────────────────────────────
# Update these paths to point to your local copies of each file.
CRW_PATH         = "data/dhw_5km.nc"
CLIM_PATH        = "data/ct5km_climatology_v3.1.nc"
BLEACHING_CSV    = "data/bleaching_confusion_labeled.csv"
ECOSTRESS_FOLDER = "data/ecostress_monthly"   # folder of masked_ECOSTRESS_LST_Median_*.tif files
MPA_SHP          = "data/Belize_MPAs_Reprojected.shp"
OUTPUT_DIR       = "figures/mpa_timeseries"   # set to None to skip saving

# ── ALERT PALETTE (NOAA CRW standard) ─────────────────────────────────────────
COLOR_WATCH   = '#FDEE66'
COLOR_WARNING = '#F2A03D'
COLOR_ALERT1  = '#D73B28'
COLOR_ALERT2  = '#9E0028'
COLOR_ALERT3  = '#5C001A'

MONTHS_LABEL = ["january","february","march","april","may","june",
                "july","august","september","october","november","december"]

# ── LOAD GLOBAL DATASETS (once) ───────────────────────────────────────────────
print("Loading global datasets...")
mpas     = gpd.read_file(MPA_SHP).to_crs("EPSG:4326")
name_col = next((c for c in mpas.columns if "name" in c.lower()), mpas.columns[0])
print(f"  MPA name column: '{name_col}' | {len(mpas)} MPAs found")

ds_crw   = xr.open_dataset(CRW_PATH)
ds_clim  = xr.open_dataset(CLIM_PATH)

df_bleach = pd.read_csv(BLEACHING_CSV)
df_bleach['time'] = pd.to_datetime(df_bleach['YearMonth'], format='%Y-%m')

# Index ECOSTRESS monthly composites, skipping 2023-06
ECOSTRESS_TIFS = {}
for tif_path in sorted(glob.glob(f"{ECOSTRESS_FOLDER}/masked_ECOSTRESS_LST_Median_*.tif")):
    if "2023-06" in tif_path:
        continue
    match = re.search(r'(\d{4}-\d{2})', tif_path)
    if match:
        ECOSTRESS_TIFS[match.group(1)] = tif_path
print(f"  ECOSTRESS composites indexed: {len(ECOSTRESS_TIFS)}")

if OUTPUT_DIR:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── HELPERS ───────────────────────────────────────────────────────────────────
def clip_to_geom(da, geom, x_dim="longitude", y_dim="latitude"):
    """Clip an xarray DataArray to a shapely geometry."""
    da = da.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim, inplace=False)
    da = da.rio.write_crs("EPSG:4326", inplace=False)
    return da.rio.clip([mapping(geom)], crs="EPSG:4326", drop=True)

def build_climatology(ds_clim, geom, lat_min, lat_max, lon_min, lon_max):
    """
    Return (bleaching_threshold, clim_df) for a given MPA geometry.
    bleaching_threshold is the MMM value; clim_df contains monthly mean SST.
    """
    use_360  = float(ds_clim.lon.max()) > 180
    lon_min_ = lon_min + 360 if (use_360 and lon_min < 0) else lon_min
    lon_max_ = lon_max + 360 if (use_360 and lon_max < 0) else lon_max
    clim_geom = translate(geom, xoff=360) if use_360 else geom

    def _mean(da, fallback_da):
        try:
            return float(clip_to_geom(da, clim_geom, x_dim="lon", y_dim="lat").mean().values)
        except Exception:
            return float(fallback_da.mean().values)

    mmm_raw = ds_clim['sst_clim_mmm'].sel(
        lat=slice(lat_max, lat_min), lon=slice(lon_min_, lon_max_))
    bleaching_threshold = _mean(mmm_raw, mmm_raw)

    monthly_means = {}
    for month in MONTHS_LABEL:
        varname = f"sst_clim_{month}"
        if varname not in ds_clim.variables:
            continue
        raw = ds_clim[varname].sel(lat=slice(lat_max, lat_min), lon=slice(lon_min_, lon_max_))
        monthly_means[month] = _mean(raw, raw)

    clim_df = pd.DataFrame({
        "Month":        [m.capitalize() for m in MONTHS_LABEL],
        "Climatology":  [monthly_means.get(m, np.nan) for m in MONTHS_LABEL],
    })
    clim_df["MonthNum"] = pd.to_datetime(clim_df["Month"], format="%B").dt.month
    clim_df.set_index("MonthNum", inplace=True)
    return bleaching_threshold, clim_df

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
for _, row in mpas.iterrows():
    mpa_name = str(row[name_col])
    geom     = row.geometry
    lon_min, lat_min, lon_max, lat_max = geom.bounds
    print(f"\n--- {mpa_name} ---")

    # 1. Climatology & MMM threshold
    bleaching_threshold, clim_df = build_climatology(
        ds_clim, geom, lat_min, lat_max, lon_min, lon_max)

    # 2. CRW SST & DHW time series
    kwargs = dict(time=slice("2023-05-01", "2024-04-30"),
                  latitude=slice(lat_max, lat_min),
                  longitude=slice(lon_min, lon_max))
    try:
        sst_clipped = clip_to_geom(ds_crw['CRW_SST'].sel(**kwargs), geom)
        dhw_clipped = clip_to_geom(ds_crw['CRW_DHW'].sel(**kwargs), geom)
    except Exception:
        sst_clipped = ds_crw['CRW_SST'].sel(**kwargs)
        dhw_clipped = ds_crw['CRW_DHW'].sel(**kwargs)

    sst_series   = sst_clipped.mean(dim=["latitude", "longitude"]).to_series()
    dhw_series   = dhw_clipped.mean(dim=["latitude", "longitude"]).to_series()
    sst_monthly  = sst_series.resample('MS').mean()
    aligned_clim = pd.Series(
        sst_monthly.index.month.map(clim_df['Climatology']).values,
        index=sst_monthly.index)

    # 3. Bleaching surveys within MPA
    df_mpa = df_bleach.copy()
    if 'Latitude' in df_mpa.columns and 'Longitude' in df_mpa.columns:
        gdf_b  = gpd.GeoDataFrame(df_mpa,
                     geometry=gpd.points_from_xy(df_mpa['Longitude'], df_mpa['Latitude']),
                     crs="EPSG:4326")
        df_mpa = gdf_b[gdf_b.geometry.within(geom)].drop(columns='geometry')
    else:
        lat_col_ = next((c for c in df_mpa.columns if 'lat' in c.lower()), None)
        lon_col_ = next((c for c in df_mpa.columns if 'lon' in c.lower()), None)
        if lat_col_ and lon_col_:
            df_mpa = df_mpa[
                (df_mpa[lat_col_] >= lat_min) & (df_mpa[lat_col_] <= lat_max) &
                (df_mpa[lon_col_] >= lon_min) & (df_mpa[lon_col_] <= lon_max)]
    print(f"  Bleaching surveys: {len(df_mpa)}")

    bleach_data   = df_mpa.groupby('time')['Bleached_Percent'].apply(list).reindex(
                        sst_monthly.index.intersection(
                            df_mpa.groupby('time').size().index))
    bleach_counts = df_mpa.groupby('time').size().reindex(bleach_data.index)

    # 4. ECOSTRESS monthly composites
    eco_ts = pd.Series(dtype=float)
    timestamps, values = [], []
    for date_str, tif_path in ECOSTRESS_TIFS.items():
        try:
            raster  = rioxarray.open_rasterio(tif_path).rio.reproject("EPSG:4326")
            clipped = raster.rio.clip([mapping(geom)], crs="EPSG:4326", drop=True)
            val     = clipped.where(clipped > 0).mean().item()
            timestamps.append(pd.to_datetime(date_str))
            values.append(val)
        except Exception as e:
            print(f"  ECOSTRESS skip {date_str}: {e}")
    if timestamps:
        eco_ts = pd.Series(values, index=timestamps).reindex(sst_monthly.index)
    eco_ts_interp = eco_ts.interpolate(method='linear')

    # 5. Plot
    fig, ax1 = plt.subplots(figsize=(16, 9))

    # DHW heat-stress shading
    above = sst_series > bleaching_threshold
    for dhw_min, dhw_max, color in [
        (0,  4,  COLOR_WATCH),
        (4,  8,  COLOR_WARNING),
        (8,  12, COLOR_ALERT1),
        (12, 16, COLOR_ALERT2),
        (16, 20, COLOR_ALERT3),
    ]:
        mask = above & (dhw_series >= dhw_min)
        if dhw_max < 20:
            mask &= (dhw_series < dhw_max)
        ax1.fill_between(dhw_series.index, bleaching_threshold, sst_series,
                         where=mask, color=color, alpha=0.7,
                         zorder=list([0,4,8,12,16]).index(dhw_min) + 1)

    ln_thresh = ax1.axhline(y=bleaching_threshold, color='dodgerblue', linestyle='--',
                            label=f'MMM Bleaching Threshold ({bleaching_threshold:.2f}°C)', zorder=5)
    ln_clim,  = ax1.plot(aligned_clim.index,  aligned_clim.values,
                         label="Climatological Mean SST", marker='o',
                         linestyle='--', color='gray', zorder=5)
    ln_sst,   = ax1.plot(sst_monthly.index, sst_monthly.values,
                         label="Observed Mean SST (CRW)", marker='o',
                         color='tab:blue', zorder=5)
    ln_eco,   = ax1.plot(eco_ts_interp.index, eco_ts_interp.values,
                         label="ECOSTRESS LST (Median)", linestyle=':', color='green', zorder=5)
    ax1.plot(eco_ts.dropna().index, eco_ts.dropna().values,
             marker='^', linestyle='none', color='green', zorder=6)

    ax1.set_xlabel("Month", fontsize=12)
    ax1.set_ylabel("Sea Surface Temperature (°C)", color='tab:blue', fontsize=14)
    ax1.set_ylim(25, 50)
    ax1.tick_params(axis='y', labelcolor='tab:blue', labelsize=12)

    ax2 = ax1.twinx()
    ax2.set_ylabel("Bleached Coral (%)", color='tab:red', fontsize=14)
    ax2.tick_params(axis='y', labelcolor='tab:red', labelsize=12)
    ax2.set_ylim(-15, 105)

    if len(bleach_data) > 0:
        box_pos = [mdates.date2num(d) for d in bleach_data.index]
        bp = ax2.boxplot(bleach_data.values, positions=box_pos,
                         widths=15, patch_artist=True, showfliers=False)
        for patch in bp['boxes']:
            patch.set_facecolor('tab:red'); patch.set_alpha(0.5)
        for median in bp['medians']:
            median.set_color('black'); median.set_linewidth(2)
        for i, date in enumerate(bleach_data.index):
            ax2.text(mdates.date2num(date),
                     max(bleach_data.iloc[i]) + 2,
                     f"n={bleach_counts.iloc[i]}",
                     ha='center', va='bottom', fontsize=10)
    else:
        ax2.text(0.5, 0.5, "No bleaching survey data\nwithin this MPA",
                 transform=ax2.transAxes, ha='center', va='center',
                 fontsize=12, color='gray', style='italic')

    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax1.tick_params(axis='x', labelsize=12)
    fig.autofmt_xdate(rotation=0, ha='center')

    ax1.legend(handles=[
        ln_clim, ln_sst, ln_eco, ln_thresh,
        Patch(facecolor='tab:red',      alpha=0.5, label='Bleaching Distribution (%)'),
        Patch(facecolor=COLOR_WATCH,    alpha=0.7, label='Watch/Warning (0 < DHW < 4)'),
        Patch(facecolor=COLOR_WARNING,  alpha=0.7, label='Alert Level 1 (4 ≤ DHW < 8)'),
        Patch(facecolor=COLOR_ALERT1,   alpha=0.7, label='Alert Level 2 (8 ≤ DHW < 12)'),
        Patch(facecolor=COLOR_ALERT2,   alpha=0.7, label='Alert Level 3 (12 ≤ DHW < 16)'),
        Patch(facecolor=COLOR_ALERT3,   alpha=0.7, label='Alert Level 4 (16 ≤ DHW < 20)'),
    ], loc='upper right', fontsize=11)

    ax1.set_title(f"Temperature & Coral Bleaching — {mpa_name} (2023–2024)", fontsize=16)
    fig.tight_layout()

    if OUTPUT_DIR:
        safe_name = re.sub(r'[^\w\-]', '_', mpa_name)
        fig.savefig(f"{OUTPUT_DIR}/{safe_name}.png", dpi=150, bbox_inches='tight')
        print(f"  Saved → {OUTPUT_DIR}/{safe_name}.png")

    plt.show()
    plt.close(fig)

print("\nAll MPAs processed.")
